# 02 — Preserve patient assignments and fit preprocessing

The four existing filenames are retained. This revision replaces the previous feature-ranking experiment with the fixed V63 business feature list. The folder name is retained for compatibility; no feature selection runs here. Do not regenerate these notebooks with the older experiment build script or use the old folder README as this revision's run guide.

**Confirmed in this revision:** supplied source-table locations, date-window helpers and the fixed V63 feature encoding are embedded. MODEL_TYPE.FEATURES is reconciled with FINAL_MODEL ordered by SEQ; frozen VALUE_P/VAR_TYP are loaded from FEATURES_SUMMARY and fingerprinted. Counts and ratios may legitimately be encoded as Binary. No percentile fitting, type inference, RND threshold, upstream sampling or feature selection runs here. Existing MODEL_DATA values are already encoded and are not encoded again.

**Remaining source blockers:** column-level claim mappings/counting grain, custom formulas, patient observation coverage and historical source availability are not fully supplied. The audit states these gaps and notebook 01 stops before inventing sequences or training. The current-state provider/plan/claim sources are not historical as-of dimensions. A claim date does not prove when that record became available. Full reconstruction status and actual metrics require the approved data runtime; no real temporal results are claimed from local checks.

Existing population: 23,151 patient snapshots, 12,447 patients, 1,345 positives; PATIENT_ID + END_DT; labels copied unchanged from the frozen source. The model-type FEATURES array supplies all 49 predictors in stored order. Calendar position 0 is newest; monthly positions 0–11 cover the original calendar buckets, with the current month truncated at END_DT. Quarterly positions 0–3 group exactly those buckets into consecutive three-month periods, not calendar-year quarters. Historical rolling windows can require source records earlier than this displayed 12-bucket sequence.

Architecture and optimizer remain the original temporal Transformer (128 width, 4 heads, 2 layers, FF256, dropout .2; weighted BCE, AdamW, validation-AP checkpointing). Raw historical features receive the supplied V63 encoding exactly once: AGE = CEIL(raw/10); Binary = raw>0; Numeric = 1 if raw>VALUE_P, else raw/(VALUE_P+1). Then the existing 49-feature model's median/mean/std preprocessing is fitted on valid TRAIN timesteps only. Padded rows are excluded from fitted statistics and re-zeroed afterwards. No feature is silently imputed to resolve missing reconstruction logic.

Frozen V63 caps and types were originally fitted using RESP; their upstream fitting population has not been checked against these held-out assignments. Reusing them preserves V63 encoding but cannot establish leakage-free held-out evaluation. All notebook reports carry this limitation. Learned preprocessing in this experiment still uses TRAIN only; the existing cohort, frozen END_DT and RESP remain unchanged. The original 250K negative sampling and dynamic LAST_DAY(MAX(service_date)) are lineage context, not instructions to resample or update the frozen population.

Only aggregate reports, tensors and model checkpoints are saved in the existing private warehouse pattern. No custom CSV/download export is provided. Clear all outputs before committing executed notebooks. TEST has previously been inspected and remains a retrospective check. Monthly/quarterly and masking are hypotheses; this two-model comparison alone does not isolate a causal masking effect from representation/preprocessing changes.


In [ ]:
# Connection and version identifiers
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply the existing private sf_options connection on the approved Spark runtime.')
sf_options_dl_poc = dict(sf_options)
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'H002'
RUN_ID = 'M001'
import re
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use short uppercase identifiers; use matching IDs in all four notebooks.')
# New output names prevent collision with prior saved models; original inputs remain unchanged.
EXPERIMENT_PREFIX = PREFIX + '_BUSINESS49_TEMPORAL_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
REFERENCE_MODEL_TABLE = PREFIX + '_MODEL_RUN_001'
REFERENCE_NAMES = {'checkpoint.pt', 'training_summary.json', 'training_history.csv', 'training_history.png'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device='auto')
print('Fixed V63 business features; source cohort and RESP retained. Run MONTHLY then QUARTERLY.')

IMPLEMENTATION_SHA256 = '65157247856a851f74cd2b8d1e34e15f003120e6e2b5ca1fe0bfb62269010b27'


In [ ]:
# Embedded input, split and preprocessing checks
"""Embedded notebook helpers: fixed business features, calendar grids and padding."""
import io
import json
import hashlib
import re
import marshal
import numpy as np
import pandas as pd


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ordered_features():
    features, comparison = configured_features()
    require(len(features) == 49, 'V63 MODEL_TYPE.FEATURES must contain exactly 49 predictors.')
    return features, comparison


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def sequence_grid(metadata, representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    rows = []
    for r in metadata.itertuples():
        cutoff = pd.Timestamp(r.END_DT)
        month = cutoff.to_period('M')
        for step in range(12 // width):
            start = (month - width * step - (width - 1)).start_time.normalize()
            end = min(cutoff, (month - width * step).end_time.normalize())
            rows.append((r.PATIENT_ID, r.END_DT, step, start.strftime('%Y-%m-%d'),
                         end.strftime('%Y-%m-%d'), int(r.RESP)))
    grid = pd.DataFrame(rows, columns=['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'RESP'])
    require(not grid.duplicated(['PATIENT_ID', 'END_DT', 'TIME_STEP']).any(), 'Duplicate sequence keys.')
    require(grid.PERIOD_END.le(grid.END_DT).all(), 'Future period boundary.')
    return grid


def verify_periods(monthly, quarterly):
    keys = ['PATIENT_ID', 'END_DT']
    for frame, count in ((monthly, 12), (quarterly, 4)):
        require(not frame.duplicated(keys + ['TIME_STEP']).any(), 'Duplicate sequence row.')
        require(frame.groupby(keys).TIME_STEP.apply(lambda x: sorted(x) == list(range(count))).all(),
                'Missing or invalid sequence positions.')
    for q in range(4):
        m = monthly.loc[monthly.TIME_STEP.between(q * 3, q * 3 + 2)]
        bounds = m.groupby(keys).agg(PERIOD_START=('PERIOD_START', 'min'), PERIOD_END=('PERIOD_END', 'max'))
        actual = quarterly.loc[quarterly.TIME_STEP.eq(q)].set_index(keys)[['PERIOD_START', 'PERIOD_END']].sort_index()
        require(bounds.sort_index().equals(actual), 'Monthly and quarterly calendar periods differ.')


def reconcile_dictionary(features, dictionary):
    rows, assigned = [], set()
    for order, name in enumerate(features):
        exact = [r for r in dictionary if r['name_complete'] and r['name'] == name]
        candidates = exact or [r for r in dictionary if not r['name_complete'] and name.startswith(r['name'])]
        match = candidates[0] if len(candidates) == 1 else None
        if match:
            require(match['seq'] not in assigned, 'Dictionary entry matched multiple model features; confirm full names.')
            assigned.add(match['seq'])
        rows.append({'FEATURE_ORDER': order, 'FEATURE_NAME': name,
                     'DICTIONARY_SEQ': match['seq'] if match else None,
                     'MATCH': ('EXACT_NAME' if exact else 'UNIQUE_VISIBLE_PREFIX') if match else 'UNRESOLVED',
                     'BUSINESS_DEFINITION': match['definition'] if match else 'No unambiguous dictionary match.',
                     'FEATURE_TYPE': (match['type_label'] + ' (dictionary label; not a casting rule)') if match else 'UNRESOLVED',
                     'DEFAULT_STATUS': match['default_status'] if match else 'UNRESOLVED',
                     'NOTES': ((match.get('notes', '') + ('; clipped name matched by unique visible prefix' if not exact else ''))
                               if match else 'Confirm full feature name/definition against V63.')})
    unmatched = pd.DataFrame([r for r in dictionary if r['seq'] not in assigned])
    return pd.DataFrame(rows), unmatched


def reconstruction_audit(features, dictionary, rules, parameters):
    matched, extra = reconcile_dictionary(features, dictionary)
    require(not set(rules).difference(features), 'Historical rules include non-V63 features.')
    rows = []
    for row in matched.to_dict('records'):
        name = row['FEATURE_NAME']
        rule = rules.get(name)
        source, logic = feature_lineage_notes(name)
        parameter = parameters[row['FEATURE_ORDER']]
        require(parameter['FEATURES'] == name, 'Audit and fixed parameter order differ.')
        row['DICTIONARY_TYPE'] = row['FEATURE_TYPE']
        row['FEATURE_TYPE'] = parameter['VAR_TYP'] + ' (fixed V63 output encoding)'
        row['VALUE_P'] = parameter['VALUE_P']
        row['VALUE_TRANSFORMATION'] = parameter['TRANSFORM']
        row['SOURCE/CALCULATION'] = source
        row['HISTORICAL_RECONSTRUCTION_STATUS'] = row.pop('DEFAULT_STATUS')
        row['HISTORICAL_LOGIC'] = logic
        row['NOTES'] += '; Frozen VAR_TYP/VALUE_P applied once after raw reconstruction; patient coverage and source availability still require evidence.'
        if rule:
            status = rule.get('status')
            require(status in ('EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'), 'Invalid reconstruction status.')
            row['HISTORICAL_RECONSTRUCTION_STATUS'] = status
            row['SOURCE/CALCULATION'] = rule.get('source', '')
            row['HISTORICAL_LOGIC'] = rule.get('logic', '')
            row['NOTES'] += '; ' + rule.get('notes', '')
            if status in ('EXACT', 'APPROXIMATED'):
                require(all(rule.get(k) for k in ('source', 'logic', 'evidence', 'observation_logic')),
                        name + ': source, calculation evidence and observation logic are required.')
                require(callable(rule.get('builder')), name + ': executable historical calculation is missing.')
                if status == 'APPROXIMATED':
                    require(bool(rule.get('approximation')), name + ': describe the approximation explicitly.')
        rows.append(row)
    audit = pd.DataFrame(rows)
    counts = audit.HISTORICAL_RECONSTRUCTION_STATUS.value_counts().reindex(
        ['EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'], fill_value=0)
    require(len(audit) == int(counts.sum()) == 49, 'Reconstruction counts must sum to 49.')
    return audit, counts, extra


def require_reconstruction(audit, rules):
    blocked = audit.loc[~audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']), 'FEATURE_NAME'].tolist()
    require(not blocked, 'Historical reconstruction blocked. Supply verified V63 calculations and coverage for: ' + ', '.join(blocked))
    require(set(rules) == set(audit.FEATURE_NAME), 'Exactly 49 historical rules are required.')


def construct_sequence(grid, features, audit, rules, representation, encoding_contract):
    require_reconstruction(audit, rules)
    keys = ['PATIENT_ID', 'END_DT', 'TIME_STEP']
    # Builders receive dates and identifiers only, never RESP or split membership.
    request = grid.drop(columns='RESP').copy()
    values, observed = [], []
    provenance = []
    for feature in features:
        rule = rules[feature]
        result = rule['builder'](request.copy(), representation)
        needed = keys + ['VALUE', 'IS_OBSERVED', 'MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE',
                         'OBSERVATION_EVIDENCE', 'PROVENANCE_KIND', 'PROVENANCE_NOTE']
        require(isinstance(result, pd.DataFrame) and set(needed).issubset(result.columns), feature + ': incomplete historical output.')
        result = result[needed].copy()
        require(not result[keys].isna().any().any() and not result.duplicated(keys).any(), feature + ': invalid historical keys.')
        require(len(result) == len(grid), feature + ': historical output must cover every requested position explicitly.')
        aligned = request.merge(result, on=keys, how='left', validate='one_to_one', indicator=True)
        require(aligned._merge.eq('both').all(), feature + ': missing historical keys.')
        require(aligned.IS_OBSERVED.isin([0, 1, False, True]).all(), feature + ': observation status is required for every position.')
        known = aligned.IS_OBSERVED.astype(bool).to_numpy()
        require(aligned.OBSERVATION_EVIDENCE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(),
                feature + ': availability must be evidenced, including unavailable periods.')
        for field in ('MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE'):
            dates = pd.to_datetime(aligned[field], errors='raise')
            require(dates.dt.tz is None and dates.dropna().eq(dates.dropna().dt.normalize()).all(),
                    feature + ': provenance dates must be exact dates; timestamp rules require explicit review.')
            require((dates.isna() | dates.le(pd.to_datetime(aligned.PERIOD_END))).all(), feature + ': future information detected in ' + field)
        numbers = pd.to_numeric(aligned.VALUE, errors='raise').to_numpy(dtype=np.float64)
        require(np.isfinite(numbers[known]).all(), feature + ': observed values must be finite; explicitly calculate observed zeros.')
        require(np.isnan(numbers[~known]).all(), feature + ': unavailable feature history must remain null before padding.')
        kinds = aligned.PROVENANCE_KIND
        require(kinds.isin(['EVENT_DERIVED', 'OBSERVED_EMPTY', 'STATIC', 'UNAVAILABLE']).all(), feature + ': invalid provenance kind.')
        require(aligned.PROVENANCE_NOTE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(), feature + ': missing provenance explanation.')
        require(np.array_equal(kinds.ne('UNAVAILABLE').to_numpy(), known), feature + ': provenance and availability disagree.')
        event_rows = kinds.eq('EVENT_DERIVED')
        require(aligned.loc[event_rows, ['MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE']].notna().all().all(),
                feature + ': event-derived values need both event and availability date maxima.')
        require(np.all(numbers[kinds.eq('OBSERVED_EMPTY')] == 0), feature + ': observed-empty provenance requires a defined zero value.')
        if kinds.eq('STATIC').any():
            require(rule.get('static_feature') is True and bool(rule.get('static_rationale')),
                    feature + ': static provenance requires a verified static-feature rule and rationale.')
        values.append(numbers)
        observed.append(known)
        provenance.append({'feature': feature, 'rule': {k: v for k, v in rule.items() if k != 'builder'},
                           'builder_sha256': hashlib.sha256(marshal.dumps(rule['builder'].__code__)).hexdigest(),
                           'observed_positions': int(known.sum())})
    raw = np.column_stack(values)
    known = np.column_stack(observed)
    # A token is fully observed only when all fixed 49 inputs are supported at this cutoff.
    # Partial feature availability is reported, not disguised as no activity.
    valid = known.all(axis=1)
    long = grid.copy()
    long[features] = raw
    long['AVAILABLE_FEATURE_COUNT'] = known.sum(axis=1)
    long['IS_VALID_TIMESTEP'] = valid.astype('int64')
    long['IS_PADDED'] = (~valid).astype('int64')
    long['PADDING_REASON'] = np.where(valid, '', 'Insufficient evidenced history for one or more fixed features')
    # Preserve raw missingness for review; only the model tensor receives padding zeros.
    n_steps = 12 if representation == 'MONTHLY' else 4
    encoded = transform_fixed_v63(raw, features, encoding_contract['parameters'], encoding_contract['parameter_sha256'])
    require(np.array_equal(np.isnan(encoded), ~known), 'Business encoding changed historical availability.')
    X = np.where(valid[:, None], encoded, 0).astype(np.float32).reshape(-1, n_steps, 49)
    mask = valid.reshape(-1, n_steps)
    require(np.isfinite(X).all(), 'NaN/Inf after padding.')
    require(np.array_equal(mask, long.IS_VALID_TIMESTEP.to_numpy().reshape(mask.shape)), 'Mask alignment failure.')
    observed_zero = known.all(axis=1) & np.all(raw == 0, axis=1)
    require(valid[observed_zero].all(), 'Observed zero activity was incorrectly masked.')
    return {'X': X, 'valid': mask, 'long': long, 'known': known, 'encoded': encoded, 'provenance': provenance,
            'representation': representation}


def sparsity_report(bundle, features):
    raw = bundle['long'][features].to_numpy(dtype=float)
    known = bundle['known']
    valid = bundle['valid']
    counts = known.sum(axis=0)
    zeros = ((raw == 0) & known).sum(axis=0)
    feature = pd.DataFrame({'FEATURE_NAME': features, 'OBSERVED_VALUES': counts,
                            'UNAVAILABLE_VALUES': (~known).sum(axis=0), 'OBSERVED_ZERO_VALUES': zeros,
                            'OBSERVED_ZERO_PERCENT': np.divide(100. * zeros, counts, out=np.full(49, np.nan), where=counts > 0)})
    encoded_zeros = ((bundle['encoded'] == 0) & known).sum(axis=0)
    feature['V63_ENCODED_ZERO_PERCENT_OBSERVED'] = np.divide(
        100. * encoded_zeros, counts, out=np.full(49, np.nan), where=counts > 0)
    eligible = int(known.sum())
    report = {'MODEL': bundle['representation'], 'TOTAL_TIMESTEPS': int(valid.size),
              'VALID_TIMESTEPS': int(valid.sum()), 'PADDED_TIMESTEPS': int((~valid).sum()),
              'VALID_PERCENT': float(valid.mean() * 100), 'PADDED_PERCENT': float((~valid).mean() * 100),
              'ALL_PADDED_SNAPSHOTS': int((~valid.any(axis=1)).sum()),
              'OBSERVED_FEATURE_ZERO_PERCENT': float(((raw == 0) & known).sum() * 100 / eligible) if eligible else None,
              'V63_ENCODED_FEATURE_ZERO_PERCENT_OBSERVED': float(encoded_zeros.sum() * 100 / eligible) if eligible else None,
              'TENSOR_ZERO_PERCENT_INCLUDING_PADDING': float((bundle['X'] == 0).mean() * 100),
              'OBSERVED_ZERO_TIMESTEPS': int((valid.reshape(-1) & np.all(raw == 0, axis=1)).sum())}
    distribution = pd.Series(valid.sum(axis=1)).value_counts().sort_index().rename_axis('VALID_TIMESTEPS').reset_index(name='SNAPSHOTS')
    return report, feature, distribution


def display_sequence(bundle, features):
    frame = bundle['long'].copy()
    if bundle['representation'] == 'MONTHLY':
        frame = frame.rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'})
    else:
        frame = frame.rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'})
    print(bundle['representation'], 'raw historical values; unavailable values are null here and zero-padded only in the model tensor')
    for label in (0, 1):
        sample = frame.loc[frame.RESP.eq(label)].head(24)
        if not sample.empty:
            print('RESP =', label)
            display(sample)
    for state in (1, 0):
        sample = frame.loc[frame.IS_VALID_TIMESTEP.eq(state)].head(12)
        print('Valid' if state else 'Padded', 'positions:', 'available' if not sample.empty else 'none in this dataset')
        if not sample.empty:
            display(sample)
    raw = bundle['long']
    summary = raw.groupby(['PATIENT_ID', 'END_DT'], sort=True).agg(
        VALID=('IS_VALID_TIMESTEP', 'sum'), PADDED=('IS_PADDED', 'sum')).reset_index()
    activity = raw[features].fillna(0).ne(0).sum(axis=1)
    summary['NONZERO_VALUES'] = activity.groupby([raw.PATIENT_ID, raw.END_DT]).sum().to_numpy()
    choices = [('relatively dense', summary.sort_values(['VALID', 'NONZERO_VALUES'], ascending=False).head(1)),
               ('partially sparse', summary.loc[summary.VALID.gt(0)].sort_values('NONZERO_VALUES').head(1)),
               ('substantial padding', summary.loc[summary.PADDED.gt(0)].sort_values('PADDED', ascending=False).head(1))]
    for title, example in choices:
        print('Structural example:', title, '(selected without model scores)')
        if example.empty:
            print('No matching example in this dataset.')
        else:
            r = example.iloc[0]
            display(frame.loc[frame.PATIENT_ID.eq(r.PATIENT_ID) & frame.END_DT.eq(r.END_DT)])


def prepared_blobs(metadata, features, bundles, audit, comparison, snapshot_X, encoding_contract):
    report = {'schema': 3, 'dataset_id': DATASET_ID, 'features': features,
              'business_encoding': encoding_contract,
              'population_sha256': digest_json(snapshot_records(metadata)),
              'snapshot_feature_sha256': array_hash(snapshot_X),
              'configuration_audit': comparison, 'audit': audit.astype(object).where(pd.notna(audit), None).to_dict('records'),
              'orientation': '0=newest; calendar buckets; most recent bucket truncated at END_DT',
              'implementation_sha256': IMPLEMENTATION_SHA256,
              'representations': {}}
    artifacts = {'population.json': canonical_json(snapshot_records(metadata)).encode()}
    for name, b in bundles.items():
        buffer = io.BytesIO()
        np.savez_compressed(buffer, X=b['X'], valid=b['valid'])
        artifacts[name + '.npz'] = buffer.getvalue()
        sparsity, _, distribution = sparsity_report(b, features)
        report['representations'][name] = {'shape': list(b['X'].shape), 'X_sha256': array_hash(b['X']),
              'valid_sha256': array_hash(b['valid']), 'sparsity': sparsity,
              'valid_distribution': distribution.to_dict('records'), 'provenance': b['provenance']}
    artifacts['manifest.json'] = canonical_json(report).encode()
    return artifacts


PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}


def load_prepared():
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require(manifest['schema'] == 3 and manifest['dataset_id'] == DATASET_ID, 'Prepared dataset ID/version changed.')
    require(manifest['implementation_sha256'] == IMPLEMENTATION_SHA256, 'Prepared dataset was produced by different code.')
    features, _ = ordered_features()
    require(features == manifest['features'], 'Authoritative feature order changed after preparation.')
    _, _, current_encoding = load_business_parameters(features)
    require(current_encoding == manifest['business_encoding'], 'Frozen V63 cap/type parameters changed after preparation.')
    metadata = pd.DataFrame(json.loads(blobs['population.json']), columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(snapshot_records(metadata)) == manifest['population_sha256'], 'Population hash mismatch.')
    audit = pd.DataFrame(manifest['audit'])
    require(audit.FEATURE_NAME.tolist() == features and audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']).all(), 'Unsupported or misordered reconstruction audit.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.shape == (len(metadata), count, 49) and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions.')
        require(np.isfinite(X).all() and np.all(X[~valid] == 0), 'Invalid padding or nonfinite input.')
        require(array_hash(X) == manifest['representations'][name]['X_sha256'] and array_hash(valid) == manifest['representations'][name]['valid_sha256'], 'Tensor/mask fingerprint changed.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    return metadata, features, bundles, manifest


def fit_temporal_preprocessor(X, valid, rows):
    observed = X[rows][valid[rows]]
    require(len(observed) > 0, 'No observed TRAIN timesteps; cannot fit preprocessing.')
    require(np.isfinite(observed).all(), 'Unresolved missingness cannot be imputed as reconstructed history.')
    state = fit_preprocessor(observed)
    return state


def transform_temporal(X, valid, state):
    values, _ = transform_features(X.reshape(-1, X.shape[-1]), state)
    values = values.reshape(X.shape)
    values[~valid] = 0
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
    reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
    metadata, hashes = bind_split(metadata, frozen, reference)
    split_statistics(metadata)
    require(hashes == audit['reference_hashes'] and digest_json(manifest) == audit['manifest_sha256'], 'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']), 'Preprocessing does not reproduce TRAIN-only fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X), 'mask_sha256': array_hash(b['valid'])})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit

def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value

def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'

def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out

def align_features(snapshots, model_data, features):
    features = parse_features(features)
    expected = normalize_metadata(snapshots).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    actual = normalize_metadata(model_data)
    missing = set(features).difference(model_data.columns)
    if missing:
        raise ValueError("Selected columns absent from MODEL_DATA: " + repr(sorted(missing)))
    source = actual.copy()
    for name in features:
        # Decimal fractions are converted to float, never through an integer cast.
        source[name] = pd.to_numeric(model_data[name], errors="raise").to_numpy(dtype=np.float64)
    aligned = expected.merge(source, on=["PATIENT_ID", "END_DT"], how="left",
                             validate="one_to_one", suffixes=("", "_SOURCE"), indicator=True)
    if not aligned._merge.eq("both").all() or not aligned.RESP.eq(aligned.RESP_SOURCE).all():
        raise ValueError("Missing source keys or conflicting labels in MODEL_DATA.")
    X = aligned[features].to_numpy(dtype=np.float64)
    if np.isinf(X).any():
        raise ValueError("Infinite source feature values.")
    return expected, X

def bind_split(metadata, frozen, reference):
    original = normalize_metadata(metadata).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized["SPLIT"] = frozen.SPLIT.to_numpy()
    normalized["SPLIT_CONFIG"] = frozen.SPLIT_CONFIG.to_numpy()
    normalized = normalized.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    if not original.equals(normalized[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Saved split differs from the prepared snapshots/labels.")
    if normalized[["SPLIT", "SPLIT_CONFIG"]].isna().any().any():
        raise ValueError("Incomplete frozen split.")
    if set(normalized.SPLIT) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    if normalized.groupby("PATIENT_ID").SPLIT.nunique().gt(1).any():
        raise ValueError("Patient leakage between splits.")
    if normalized.SPLIT_CONFIG.nunique() != 1:
        raise ValueError("Inconsistent split configuration.")
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {"snapshot_manifest_sha256": digest_json(records),
              "split_config_sha256": digest_json(json.loads(normalized.SPLIT_CONFIG.iloc[0]))}
    if reference.get("run_id") != "RUN_001" or reference.get("training_complete") is not True:
        raise ValueError("Expected completed original RUN_001 reference.")
    if any(reference.get("input_hashes", {}).get(k) != v for k, v in hashes.items()):
        raise ValueError("Patient assignments differ from original RUN_001 fingerprints.")
    for _, part in normalized.groupby("SPLIT"):
        if set(part.RESP) != {0, 1}:
            raise ValueError("Each split needs both outcome classes.")
    return normalized, hashes

def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}

def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)
def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())

def read_query(query):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load())

def fetch_source_features(features):
    fields = ["PATIENT_ID", "END_DT", "RESP"] + features
    selected = ", ".join("M." + quote_identifier(f) for f in fields)
    # Select only the frozen cohort. Duplicated source keys remain visible and fail validation.
    query = (f"SELECT {selected} FROM {DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA M "
             f"INNER JOIN (SELECT DISTINCT PATIENT_ID, END_DT FROM {DATABASE}.DS_ML.{PREFIX}_SNAPSHOTS) S "
             "ON M.PATIENT_ID = S.PATIENT_ID AND M.END_DT = S.END_DT")
    return read_query(query).toPandas()

def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")
import base64
def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]

"""Pure fixed-parameter transformations and date boundaries; no warehouse IO.

These helpers do not reconstruct a feature, choose predictors, infer source
coverage, fit percentiles, infer feature types, or select a modeling population.
"""

import hashlib
import json
import math
from collections.abc import Sequence

import numpy as np
import pandas as pd


def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(len(result) == 49, 'Exactly 49 authoritative predictors are required.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == 49,
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (49 ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    49-feature input list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)


def transform_fixed_v63(raw, features, parameters, expected_parameter_hash=None):
    """Transform any numeric array whose last axis is the fixed ordered 49.

    AGE: ceil(raw / 10).
    Binary: 1 exactly when raw > 0, else 0.
    Numeric: 1 when raw > VALUE_P, else raw / (VALUE_P + 1).
    Null raw values stay NaN. No filling, scaling fit, clipping below zero,
    percentile calculation, ranking or row/feature selection occurs here.
    """
    features = _v63_features(features)
    _v63_require(isinstance(parameters, (list, tuple)) and len(parameters) == 49,
                 'Exactly 49 frozen parameter records are required.')
    _v63_require(all(isinstance(row, dict) for row in parameters), 'Invalid parameter record.')
    required_keys = {'FEATURE_ORDER', 'FEATURES', 'VALUE_P', 'VAR_TYP', 'TRANSFORM'}
    _v63_require(all(set(row) == required_keys for row in parameters),
                 'Frozen parameter record fields changed.')
    _v63_require([row['FEATURES'] for row in parameters] == features,
                 'Parameter feature order differs from the authoritative list.')
    _v63_require([row['FEATURE_ORDER'] for row in parameters] == list(range(49)),
                 'Parameter FEATURE_ORDER must be exactly 0 through 48.')
    # Revalidate fixed values and transformation precedence; do not trust a
    # modified or hand-assembled parameter manifest merely because it has 49 rows.
    canonical, digest = fixed_v63_parameters(features, pd.DataFrame(parameters))
    _v63_require(list(parameters) == canonical, 'Frozen parameter values or transform rules changed.')
    if expected_parameter_hash is not None:
        _v63_require(isinstance(expected_parameter_hash, str) and digest == expected_parameter_hash,
                     'Frozen FEATURES_SUMMARY parameter fingerprint changed.')
    array = np.asarray(raw)
    _v63_require(array.ndim >= 1 and array.shape[-1] == 49,
                 'Input last axis must contain the 49 authoritative features.')
    _v63_require(not np.iscomplexobj(array), 'Complex raw feature values are invalid.')
    if array.dtype == object:
        _v63_require(not any(isinstance(value, (complex, np.complexfloating)) for value in array.flat),
                     'Complex raw feature values are invalid.')
        # Preserve nullable pandas/scalar missing values before numeric conversion.
        array = np.where(pd.isna(array), np.nan, array)
    try:
        values = np.array(array, dtype=np.float64, copy=True)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Raw features must contain real numbers or nulls.') from error
    _v63_require(not np.isinf(values).any(), 'Infinite raw feature values are invalid.')
    out = np.full(values.shape, np.nan, dtype=np.float64)
    for index, row in enumerate(canonical):
        column = values[..., index]
        available = ~np.isnan(column)
        if row['TRANSFORM'] == 'AGE_CEIL_DECADE':
            transformed = np.ceil(column / 10.0)
        elif row['TRANSFORM'] == 'BINARY_POSITIVE':
            transformed = (column > 0).astype(np.float64)
        else:
            cap = row['VALUE_P']
            transformed = np.where(column > cap, 1.0, column / (cap + 1.0))
        out[..., index] = np.where(available, transformed, np.nan)
    _v63_require(not np.isinf(out).any(), 'Fixed transformation produced infinite values.')
    _v63_require(np.array_equal(np.isnan(out), np.isnan(values)),
                 'Fixed transformation must preserve the raw null mask.')
    return out


def _v63_date(value, label='cutoff'):
    try:
        date = pd.Timestamp(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(label + ' must be an exact date.') from error
    _v63_require(not pd.isna(date), label + ' must be explicitly supplied.')
    _v63_require(date.tzinfo is None and date == date.normalize(),
                 label + ' must be a timezone-naive date, not an intraday timestamp.')
    return date


def v63_rolling_year_start(end):
    """DATEADD(year, -1, end) + 1 day, including leap-day clamping."""
    cutoff = _v63_date(end)
    return cutoff - pd.DateOffset(years=1) + pd.Timedelta(days=1)


def v63_recent_inclusive_window(end):
    """Inclusive DATEADD(month,-4,end) through DATEADD(month,-1,end).

    These are shifted dates, not first/last boundaries of calendar months.
    """
    cutoff = _v63_date(end)
    return cutoff - pd.DateOffset(months=4), cutoff - pd.DateOffset(months=1)


def v63_custom_l3m_inclusive_window(end):
    """Inclusive end-90 days through end (91 possible calendar dates)."""
    cutoff = _v63_date(end)
    return cutoff - pd.Timedelta(days=90), cutoff


def v63_hcp_730_day_start(end):
    """Return end-730 days; source SQL must retain its own boundary operators."""
    return _v63_date(end) - pd.Timedelta(days=730)


def v63_adherence_270_day_start(*, effective_data_end):
    """Return the supplied effective-data-end anchor minus 270 days.

    A snapshot cutoff is not an implicit substitute for effective_data_end.
    This helper creates a date boundary only; it does not calculate adherence.
    """
    return _v63_date(effective_data_end, 'effective_data_end') - pd.Timedelta(days=270)


"""User-supplied V63 lineage and frozen business encodings; no feature selection."""
SOURCE_REGISTRY = [
    ('MEDICAL', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.MEDICAL_EVENTS_LATEST', 'SERVICE_DATE', 'DX/PX claims; current extract, no historical availability timestamp supplied'),
    ('PHARMACY', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PHARMACY_EVENTS_LATEST', 'FILL_DATE', 'RX claims; transaction and counting rules still require column-level mapping'),
    ('PROVIDERS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PROVIDERS_LATEST', '', 'Current specialty/type; not a historical as-of dimension'),
    ('PLANS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PLANS_LATEST', '', 'Current insurance group/segment; not a historical as-of dimension'),
    ('DEMOGRAPHICS', 'DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719.PATIENT_DEMOGRAPHICS_LATEST', '', 'Year of birth for year-based age; patient/YOB columns must be verified'),
    ('CODE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID', '', 'Code descriptions, hierarchy and chronic flags'),
    ('ANNUAL_CODE_COUNTS', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID_ANNUAL_CNT', '', 'Upstream inclusion lineage only; do not rerun selection'),
    ('PROCEDURE_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.ALL_PROCEDURES_SIMPLE', '', 'ICD-10-PCS decomposition; not a substitute for visit-sequence features'),
    ('HCP_REFERENCE', 'DSVC_TAKEDA_TA_PRIVATE.GATREX_TRIGGER.HCP_LOOKUP', '', 'Active NPI/specialty; historical version not supplied'),
    ('DRUG_BASKET', 'TAK861.NARCOLEPSY_MARKET_BASKET_CODES', '', 'Database not supplied; code columns and inclusion rules still required'),
    ('UPSTREAM_UNIVERSE', 'TAK861.TAK861_TX_READY_PATIENT_UNIVERSE_V8_CF', '', 'Lineage only; never substitute for frozen V63 snapshots'),
    ('UPSTREAM_BACKTEST', 'TAK861.TAK861_TX_READY_BACKTEST_UNIVERSE_V8_CF', '', 'Lineage only; never replace the existing TEST assignment'),
]


def inspect_source_schemas():
    rows = []
    for kind, table, date_column, note in SOURCE_REGISTRY:
        if table.count('.') != 2:
            rows.append({'SOURCE': kind, 'TABLE': table, 'COLUMN': None, 'DATA_TYPE': None,
                         'STATUS': 'DATABASE_UNRESOLVED', 'NOTES': note})
            continue
        try:
            schema = read_table(table).schema
            rows.extend({'SOURCE': kind, 'TABLE': table, 'COLUMN': field.name,
                         'DATA_TYPE': field.dataType.simpleString(), 'STATUS': 'SCHEMA_READ', 'NOTES': note}
                        for field in schema.fields)
        except Exception as error:
            # A metadata-discovery failure is displayed; it cannot enable reconstruction.
            # Do not print a connector exception that might contain connection details.
            rows.append({'SOURCE': kind, 'TABLE': table, 'COLUMN': None, 'DATA_TYPE': None,
                         'STATUS': type(error).__name__, 'NOTES': note + '; metadata unavailable on this runtime'})
    return pd.DataFrame(rows)


def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'max_f_50': 'Ceiling only; RND explanation remains a hypothesis until ranked rows establish it'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    required = set(['PATIENT_ID', 'END_DT', 'RESP'] + features)
    require(required.issubset(read_table(SOURCE_PREFIX + '_MODEL_DATA').columns), 'MODEL_DATA lacks configured columns.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'raw reconstructed features', 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract


def display_feature_sources(features, summary, parameters):
    allowed = ['FEATURES', 'SEQ', 'F_FLG', 'VAR_TYP', 'VALUE_P', 'VALUE_CNT', 'RVALUE', 'RR_RATIO', 'IMPORTANCE', 'IMPORTANCE_MOD']
    selected = pd.DataFrame({'FEATURES': features, 'FEATURE_ORDER': range(49)}).merge(
        summary[[c for c in allowed if c in summary.columns]], on='FEATURES', how='left', validate='one_to_one')
    display(selected)
    if {'SEQ', 'F_FLG'}.issubset(summary.columns):
        seq = pd.to_numeric(summary.SEQ, errors='raise')
        ranked = summary.loc[seq.le(50), [c for c in allowed if c in summary.columns]].copy()
        ranked['SEQ'] = seq.loc[ranked.index]
        ranked['IN_FIXED_49'] = ranked.FEATURES.isin(features)
        print('Original top-50 metadata for reconciliation only; no feature selection is performed:')
        display(ranked.sort_values('SEQ'))
        dropped = ranked.loc[ranked.F_FLG.astype(str).str.lower().eq('dropped')]
        print('Dropped rows within the original ceiling (do not assume RND is the dictionary discrepancy):')
        display(dropped)
    print('Counts/ratios can legitimately have Binary encoding: VAR_TYP is the learned V63 output type.')
    print('Frozen VALUE_P/VAR_TYP are reused, never fitted with current validation/test labels; their upstream fit population is unverified.')


def documented_feature_windows(request):
    out = request[['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END']].copy()
    # This is a date-boundary display, not evidence of patient observation coverage.
    boundaries = []
    for value in request.PERIOD_END:
        recent_start, recent_end = v63_recent_inclusive_window(value)
        l3m_start, l3m_end = v63_custom_l3m_inclusive_window(value)
        boundaries.append({
            'PROCEDURE_L12M_START_INCLUSIVE': v63_rolling_year_start(value).strftime('%Y-%m-%d'),
            'PROCEDURE_L12M_END_INCLUSIVE': value,
            'GENERIC_RECENT_START_INCLUSIVE': recent_start.strftime('%Y-%m-%d'),
            'GENERIC_RECENT_END_INCLUSIVE': recent_end.strftime('%Y-%m-%d'),
            'CUSTOM_L3M_START_INCLUSIVE': l3m_start.strftime('%Y-%m-%d'),
            'CUSTOM_L3M_END_INCLUSIVE': l3m_end.strftime('%Y-%m-%d'),
            'HCP_AT_730_DAY_START_INCLUSIVE': v63_hcp_730_day_start(value).strftime('%Y-%m-%d'),
            'ADHERENCE_ANCHOR': 'effective_data_end not supplied; do not substitute END_DT',
        })
    for key in boundaries[0] if boundaries else []:
        out[key] = [row[key] for row in boundaries]
    return out


def feature_lineage_notes(name):
    if name == 'AGE':
        return ('DEMOGRAPHICS: patient year of birth', 'Raw age = cutoff year minus birth year; encoded AGE = CEIL(raw age/10). YOB source column/availability must be verified.')
    if name.startswith('MAX_AT_') or name.startswith('AVG_AT_'):
        return ('PHARMACY + PROVIDERS/HCP_REFERENCE', 'Provider AT usage lookback: cutoff minus 730 days through cutoff. Patient-to-HCP linkage, denominator, NTILE population/averaging and drug exclusions remain unresolved.')
    if name.startswith('_'):
        return ('Custom visit-sequence logic not established', 'Dictionary visit sequences are not ICD-10-PCS character decomposition; visit grain, order and within-five-visits rule are still required.')
    if name in {'TIMES_GENERIC_MIX_ADJUSTED', 'UNIQUE_GENERICS_TRIED', 'NUM_DISCONTINUATIONS', 'NUM_GAPS_30_PLUS_DAYS'}:
        return ('PHARMACY + DRUG_BASKET; custom feature branch', 'Custom lifetime/adherence/treatment logic coexists with procedure features; 270-day adherence anchor is effective_data_end, not automatically END_DT. Exact event/gap/overlap/generic rules remain unresolved.')
    if name.endswith('_L3M'):
        return ('MEDICAL + CODE_REFERENCE; custom feature branch', 'Custom L3M: cutoff minus 90 days through cutoff, inclusive. Diagnosis sets and counting/deduplication grain must be verified.')
    if name.endswith('_L12M') or name == 'L12M_NARCO_CLAIMS':
        return ('MEDICAL + CODE_REFERENCE; rolling feature', 'L12M: DATEADD(year,-1,cutoff)+1 day through cutoff, inclusive. Qualifying codes and count grain still required.')
    if name == 'L6M_NARCO_CLAIMS':
        return ('MEDICAL + CODE_REFERENCE; custom feature branch', 'Six-month presence is described; exact calendar/day boundary, code set and coverage remain unresolved.')
    if name in {'NARCO_CLAIMS_RECENT_RATIO', 'SLEEP_MED_VISIT_RECENCY_PCT'}:
        return ('Custom numerator/denominator source must be verified', 'Do not substitute generic RECENT_PCT windows for a custom ratio; numerator/denominator and zero-denominator rules remain unresolved.')
    if 'RECENT_PCT' in name:
        return ('Generic recency branch or custom branch; resolve lineage', 'Generic RECENT_PCT uses DATEADD(month,-4,cutoff) through DATEADD(month,-1,cutoff); earlier description also requires value>6 and recent>2. Verify actual branch/denominator before implementing.')
    source = 'MEDICAL/CODE_REFERENCE' if name.startswith(('CPT_', 'DX_', 'PL_', 'VISIT_', 'LEVEL_')) else 'PHARMACY/CODE_REFERENCE'
    if 'SPECIALIST' in name or name.startswith('HCPS_'):
        source += ' + PROVIDERS/HCP_REFERENCE (current state)'
    if any(word in name for word in ('INSURANCE', 'PAYER')):
        source += ' + PLANS (current state)'
    return (source, 'Procedure branch uses inclusive one-year window at each cutoff; confirm branch, code mapping, qualifying statuses and count/flag grain. Current metadata is not historically versioned.')


In [ ]:
# Load the verified monthly and quarterly inputs
metadata, features, bundles, manifest = load_prepared()
print('Confirmed feature count:', len(features))
display(pd.DataFrame(manifest['business_encoding']['parameters']))
print('V63 encodings verified against source metadata; not applied again. Upstream cap/type fit population remains unverified.')
for name, b in bundles.items():
    print(name, 'V63 encoded tensor before TRAIN-only scaling:', b['raw_X'].shape, '| mask:', b['valid'].shape)


In [ ]:
# Bind both representations to the original patient split
frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
metadata, reference_hashes = bind_split(metadata, frozen, reference)
statistics = split_statistics(metadata)
display(statistics)
display(pd.DataFrame([{'TRAIN_VALIDATION_OVERLAP': 0, 'TRAIN_TEST_OVERLAP': 0, 'VALIDATION_TEST_OVERLAP': 0,
                       'MONTHLY_QUARTERLY_ASSIGNMENTS_IDENTICAL': True}]))
indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
print('Same source keys, labels and original RUN_001 patient assignments verified; no new splits created.')


In [ ]:
# Fit existing business-feature scaling on valid TRAIN timesteps only
preprocessing = {}
for name, b in bundles.items():
    state = fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train'])
    preprocessing[name] = state
    values = transform_temporal(b['raw_X'], b['valid'], state)
    require(values.shape == b['raw_X'].shape and np.all(values[~b['valid']] == 0), 'Transformed padding changed.')
    display(pd.DataFrame({'MODEL': name, 'FEATURE_NAME': features, 'TRAIN_MEDIAN': state['median'],
                          'TRAIN_MEAN': state['mean'], 'TRAIN_SCALE': state['scale']}))
    first = int(indices['train'][0])
    sample = pd.DataFrame(values[first], columns=features)
    sample.insert(0, 'TIME_STEP', range(len(sample)))
    sample.insert(0, 'END_DT', metadata.END_DT.iloc[first])
    sample.insert(0, 'PATIENT_ID', metadata.PATIENT_ID.iloc[first])
    sample['IS_VALID_TIMESTEP'] = b['valid'][first].astype(int)
    sample['IS_PADDED'] = (~b['valid'][first]).astype(int)
    print(name, 'example of standardized Transformer input')
    display(sample)


In [ ]:
# Save frozen preprocessing and the identical split manifest
split_rows = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
split_audit = {'manifest_sha256': digest_json(manifest), 'reference_hashes': reference_hashes,
               'preprocessing_sha256': digest_json(preprocessing), 'split_summary': statistics.to_dict('records'),
               'fit_scope': 'Valid TRAIN timesteps only, separately for monthly and quarterly',
               'new_assignments_created': False, 'monthly_quarterly_assignments_identical': True}
save_artifacts(SPLIT_TABLE, {'split.json': canonical_json(split_rows).encode(),
    'preprocessing.json': canonical_json(preprocessing).encode(), 'audit.json': canonical_json(split_audit).encode()})
experiments, _, _ = load_experiment()
print('Read-back verified. Continue with notebook 03.')
